In [4]:
%pip install networkx
%pip install langdetect
%pip install pyarrow
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
    --------------------------------------- 0.2/10.0 MB 7.3 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/10.0 MB 12.2 MB/s eta 0:00:01
   -------- ------------------------------- 2.2/10.0 MB 17.1 MB/s eta 0:00:01
   ---------------- ----------------------- 4.1/10.0 MB 23.5 MB/s eta 0:00:01
   ------------------------------ --------- 7.5/10.0 MB 34.4 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 39.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------- ----------------------- 5.2/12.6 MB 110.5 MB/s eta 0:00:01
   ------------------------------- -------- 9.8/12.6 MB 104.1 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.6 MB 93.9 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 81.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/348.2 kB ? eta -:--:--
   --


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from pathlib import Path
import pandas as pd
import networkx as nx
from langdetect import detect

In [7]:
path = Path.cwd().parent / 'dataset' / 'twcs.csv' 
df = pd.read_csv(path)

In [8]:
print(df.shape)
print(df.columns.tolist())
print(df.head())

(2811774, 7)
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id

In [9]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [10]:
# Number of unique customer accounts interacting with each brand
customer_counts = (
    df[df["inbound"]]
    .groupby("author_id")["tweet_id"]
    .count()
    .sort_values(ascending=False)
)

display(customer_counts.head(20))

author_id
115911    1286
120576    1010
115913     563
116230     454
169172     448
117627     406
115888     332
116136     295
116421     276
115722     252
115714     250
115990     243
115850     243
121239     210
203476     197
115725     194
115798     181
127296     176
170351     174
169916     172
Name: tweet_id, dtype: int64

In [11]:
tweet_to_author = df.set_index("tweet_id")["author_id"]

inbound = df[df["inbound"] == True].copy()
inbound["brand"] = inbound["response_tweet_id"].map(tweet_to_author)
display(inbound[["tweet_id", "author_id", "text", "response_tweet_id", "brand"]].head(10))

,tweet_id,author_id,text,response_tweet_id,brand
1,2,115712,@sprintcare and how do you propose we do that,NaN,NaN
2,3,115712,@sprintcare I have sent several private messag...,1,NaN
4,5,115712,@sprintcare I did.,4,NaN
6,8,115712,@sprintcare is the worst customer service,"9,6,10",NaN
8,12,115713,@sprintcare You gonna magically change your co...,"11,13,14",NaN
10,16,115713,@sprintcare Since I signed up with you....Sinc...,15,NaN
12,18,115713,@115714 y’all lie about your “great” connectio...,17,NaN
14,20,115715,"@115714 whenever I contact customer support, t...",19,NaN
16,22,115716,@Ask_Spectrum Would you like me to email you a...,25,NaN
18,26,115716,@Ask_Spectrum I received this from your corpor...,27,NaN


### Number of Customers per Company

In [12]:
# Separate customers and companies
customers = df[df["inbound"] == True]
companies = df[df["inbound"] == False]

In [13]:
# Merge them so every customer tweet is paired with the company that replied to it
merged_df = customers.merge(
    companies, 
    left_on="in_response_to_tweet_id", 
    right_on="tweet_id", 
    suffixes=("_cust", "_comp")
)
merged_df.head()

,tweet_id_cust,author_id_cust,inbound_cust,created_at_cust,text_cust,response_tweet_id_cust,in_response_to_tweet_id_cust,tweet_id_comp,author_id_comp,inbound_comp,created_at_comp,text_comp,response_tweet_id_comp,in_response_to_tweet_id_comp
0,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
2,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
3,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0
4,16,115713,True,Tue Oct 31 20:00:43 +0000 2017,@sprintcare Since I signed up with you....Sinc...,15,17.0,17,sprintcare,False,Tue Oct 31 19:59:13 +0000 2017,@115713 H there! We'd definitely like to work ...,16,18.0


In [14]:
# Group by the company's ID and count UNIQUE customer IDs
brand_customer_counts = (
    merged_df.groupby("author_id_comp")["author_id_cust"]
    .nunique()
    .sort_values(ascending=False)
)
print("Number of customers per company:\n")
display(brand_customer_counts.head(20))

Number of customers per company:



author_id_comp
AmazonHelp         40671
AppleSupport       23668
Uber_Support       12780
AmericanAir         9699
SpotifyCares        8723
VirginTrains        8060
Delta               8034
ATVIAssist          7702
SouthwestAir        7524
Tesco               7397
British_Airways     6523
XboxSupport         6448
TMobileHelp         6285
AskPlayStation      5693
comcastcares        5593
Ask_Spectrum        5548
sainsburys          5482
ChipotleTweets      5364
GWRHelp             5269
hulu_support        5229
Name: author_id_cust, dtype: int64

# Filter down to the amazon data rows

In [15]:
# Amazon keyword filter and author_id filter + NaN cases handled
amazon_direct_mask = (
    (df['author_id'] == 'AmazonHelp') | 
    (df['text'].str.contains('AmazonHelp', case=False, na=False))
)
brand_df = df[amazon_direct_mask]

In [16]:
# Grab the IDs of tweets involved in these interactions
tweet_ids = set(brand_df['tweet_id'].unique())

# Split comma-separated response IDs safely
response_ids = set()
for x in brand_df['response_tweet_id'].dropna().astype(str):
    response_ids.update([int(i) for i in x.split(',') if i.strip().isdigit()])

in_reply_ids = set(brand_df['in_response_to_tweet_id'].dropna().astype(int).unique())

In [17]:
# Combine them to get the complete conversation threads
all_connected_ids = tweet_ids.union(response_ids).union(in_reply_ids)
sub_df = df[df['tweet_id'].isin(all_connected_ids)].copy()

In [18]:
# Use NetworkX to instantly calculate thread relationships (connected components)
G = nx.Graph()

# Add all tweet IDs as nodes
G.add_nodes_from(sub_df['tweet_id'])

# Create edges (links) between a tweet and the tweet it is responding to
edges = sub_df[['tweet_id', 'in_response_to_tweet_id']].dropna().astype({'in_response_to_tweet_id':int})
G.add_edges_from(zip(edges['tweet_id'], edges['in_response_to_tweet_id']))

# Extract the isolated conversation groups and map them to a new column
convo_mapping = {}
for convo_idx, component in enumerate(nx.connected_components(G)):
    for tweet_id in component:
        convo_mapping[tweet_id] = convo_idx

# Apply brand new conversation IDs to final dataframe
sub_df['conversation_id'] = sub_df['tweet_id'].map(convo_mapping)

# Sort them so they read naturally like a real chat log
amazon_df = sub_df.sort_values(by=['conversation_id', 'created_at']).reset_index(drop=True)

In [19]:
display(amazon_df.iloc[20:50])

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,conversation_id
20,628,115826,True,Tue Oct 31 21:57:24 +0000 2017,@115828 How about you guys figure out my Xbox ...,626,630.0,5
21,626,AmazonHelp,False,Tue Oct 31 22:28:00 +0000 2017,@115826 I'm sorry for the wait. You'll receive...,627,628.0,5
22,627,115827,True,Wed Nov 01 12:50:18 +0000 2017,@AmazonHelp @115826 Yeah this is crazy we’re l...,629,626.0,5
23,629,AmazonHelp,False,Wed Nov 01 12:53:34 +0000 2017,@115827 Thanks for your patience. ^KM,NaN,627.0,5
24,632,115829,True,Tue Oct 31 21:34:58 +0000 2017,@115830 my package was ‘accidentally’ opened.....,631,NaN,6
25,631,AmazonHelp,False,Tue Oct 31 22:27:00 +0000 2017,@115829 I'm sorry your order arrived in this c...,NaN,632.0,6
26,634,115831,True,Tue Oct 31 21:39:58 +0000 2017,@115821 @AmazonHelp why is my order at my loca...,633,NaN,7
27,633,AmazonHelp,False,Tue Oct 31 22:26:37 +0000 2017,@115831 I'm sorry for the wait. Please reach o...,NaN,634.0,7
28,636,115832,True,Tue Oct 31 22:05:36 +0000 2017,"Thanks for the style advice, @115833 look ...I...",635,NaN,8
29,635,AmazonHelp,False,Tue Oct 31 22:26:07 +0000 2017,@115832 Alexa says both styles are working for...,NaN,636.0,8


In [20]:
amazon_df.shape

(367947, 8)

In [21]:
def is_english(text):
    try:
        # Detect language; return True only if it is English ('en')
        return detect(text) == 'en'
    except:
        return False

# Group by conversation and check the language of the very first tweet in each thread
# (Checking the first tweet is much faster than checking all 360,000+ rows individually)
first_tweets = amazon_df.groupby('conversation_id').first()
english_convo_ids = first_tweets[first_tweets['text'].apply(is_english)].index

# Filter down to only English conversations
english_amazon_df = amazon_df[amazon_df['conversation_id'].isin(english_convo_ids)].copy() 

In [22]:
english_amazon_df = english_amazon_df.reset_index(drop=True)
english_amazon_df.shape

(280081, 8)

In [23]:
english_amazon_df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,conversation_id
0,617,115820,True,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service @1158...,615,NaN,2
1,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0,2
2,616,115820,True,Tue Oct 31 23:22:08 +0000 2017,@AmazonHelp 3 different people have given 3 di...,618,615.0,2
3,618,AmazonHelp,False,Tue Oct 31 23:28:00 +0000 2017,@115820 We'd like to take a further look into ...,619,616.0,2
4,619,115820,True,Tue Oct 31 23:32:26 +0000 2017,@AmazonHelp I frankly don't have the patience ...,NaN,618.0,2


In [26]:
# Define a clean, structured output path
output_dir = Path.cwd().parent / 'dataset' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

# Save as compressed Parquet (Best practice for large data)
output_file = output_dir / 'amazon_tweets_english_v1.parquet'
english_amazon_df.to_parquet(output_file, compression='snappy', index=False)

print(f"Saved successfully.")

Saved successfully.
